In [1]:
%cd ../

/u02/thanh/workplace/plasma


In [2]:
import inspect
import plasma.data_model as pdm

from pydantic import BaseModel
from typing import NamedTuple, dataclass_transform

from plasma.data_model.base_model.field import Field, Composite, List

In [3]:
@pdm.model
class A:
    a:int
    b:int
    c:list[str]

In [4]:
@pdm.model
class B:
    b1:A
    b2:list[A]
    b3:list

In [5]:
@pdm.model
class C:
    c1:B
    c2:list[B]

In [7]:
a = C.__accessors
a

{'c1.b1.a': C.c1.b1.a,
 'c1.b1.b': C.c1.b1.b,
 'c1.b1.c': C.c1.b1.c,
 'c1.b2.@idx.a': A.a,
 'c1.b2.@idx.b': A.b,
 'c1.b2.@idx.c': A.c,
 'c1.b3': C.c1.b3,
 'c2.@idx.b1.a': B.b1.a,
 'c2.@idx.b1.b': B.b1.b,
 'c2.@idx.b1.c': B.b1.c,
 'c2.@idx.b2.@idx.a': A.a,
 'c2.@idx.b2.@idx.b': A.b,
 'c2.@idx.b2.@idx.c': A.c,
 'c2.@idx.b3': B.b3}

In [9]:
template = {}
for k in a:
    container = template
    components = k.split('.')
    for i, c in enumerate(components):
        if c == '@idx':
            container.append({})
            current = container[0]
        elif i < len(components) - 1 and components[i + 1] == '@idx':
            current = container.setdefault(c, [])
        else:
            current = container.setdefault(c, {})
        
        container = current


In [10]:
template

{'c1': {'b1': {'a': {}, 'b': {}, 'c': {}},
  'b2': [{'a': {}, 'b': {}, 'c': {}}, {}, {}],
  'b3': {}},
 'c2': [{'b1': {'a': {}, 'b': {}, 'c': {}},
   'b2': [{'a': {}, 'b': {}, 'c': {}}, {}, {}],
   'b3': {}},
  {},
  {},
  {},
  {},
  {},
  {}]}

In [ ]:
c = C(
        B(
            A(5, 6, ['a', 'b']),
            [
                A(5, 6, ['a', 'b']),
            ],
            [1, 2]
        ),
        
    )

In [ ]:
a = C.__accessors
a

In [ ]:
b = {k.replace('@idx', '1'):v for k, v in a.items()}
b

In [ ]:
from plasma.data_model.base_model.accessors import AccessorMap 

In [ ]:
c = AccessorMap().to_dict(b)
c

In [ ]:
d = AccessorMap().from_dict(c)
d

In [ ]:
pdm.ModelConstructor(B).from_fields(
    {
        B.b1:A(8, 9),
        B.b2:[A(1, 2), A(3, 4)]
    }
)

In [ ]:
b = pdm.ModelConstructor(B).from_fields(
    {
        B.b1.a:8,
        B.b1.b:9,
        B.b2: [
            {'a': 9, 'b': 10},
            {'a': 9, 'b': 11},
        ]
    }
)

In [ ]:
pdm.Validator(B)(b)

In [ ]:
pdm.Validator(B)(B(A(5, 6), [A(1, 1), A(1, [2]), 19]))